# Week 2 - AI Applications - Exercise 2: Apartment Predictor + LLM Workflow

Course-fit note: This Week 2 variant is specific to AI Applications and combines a numeric model with an LLM interaction layer.

Because this is Swiss housing data, write your prompts in German so town names such as `Zürich` match the dataset more reliably.

In this exercise, you first build and test logic in the notebook, then transfer the same functions into `app_student.py`, and promote to `app.py` for deployment.


## Learning Goals

- Load and use a saved `.pkl` model for numeric prediction
- Convert natural-language wishes into structured model input
- Build a thin LLM explanation layer around model output
- Prepare a clean transfer from notebook code to a Gradio app


In [7]:
import json
import os
import pickle
import re
import urllib.error
from pathlib import Path

import numpy as np
import pandas as pd
from openai import OpenAI

from urllib import response
from xmlrpc import client


In [8]:
DATA_PATH = Path("bfs_municipality_and_tax_data.csv")
MODEL_PATH = Path("random_forest_regression.pkl")

LLM_API_KEY = os.getenv("LLM_API_KEY", "")
LLM_MODEL = os.getenv("LLM_MODEL", "")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing file: {DATA_PATH}")

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Missing file: {MODEL_PATH}")

df_bfs_data = pd.read_csv(DATA_PATH)
df_bfs_data["tax_income"] = (
    df_bfs_data["tax_income"].astype(str).str.replace("'", "", regex=False).astype(float)
)

town_to_row = {str(row["bfs_name"]).lower(): row for _, row in df_bfs_data.iterrows()}
valid_towns = list(df_bfs_data["bfs_name"].sort_values().unique())

# Optional: create an OpenAI client if you use the OpenAI SDK for your TODOs.
# client = OpenAI(api_key=LLM_API_KEY) if LLM_API_KEY else None

print(f"Loaded towns: {len(valid_towns)}")
df_bfs_data.head(2)


Loaded towns: 2155


,bfs_number,bfs_name,pop,pop_dens,frg_pct,emp,tax_income
0,1,Aeugst am Albis,1981,250.442478,14.184755,442.0,108788.0
1,2,Affoltern am Albis,12303,1161.756374,28.700317,6920.0,72583.0


In [9]:
print(f"Key set: {bool(LLM_API_KEY)}")
print(f"Model: {LLM_MODEL}")

Key set: True
Model: gpt-5.4-mini


## Step 1 - Load The Saved `.pkl` Model

Reuse the provided scikit-learn model file instead of training a new model in this notebook.


In [10]:
# TODO: load the pickled model from MODEL_PATH
with open(MODEL_PATH, "rb") as model_file:
     model = pickle.load(model_file)
#model = None

if model is None:
    print("TODO: load model before continuing")
else:
    print(f"Loaded model type: {type(model).__name__}")


Loaded model type: RandomForestRegressor


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


## Step 2 - Town Matching Helper


In [11]:
def match_town(user_town: str):
    """Return the canonical town name from the dataset, or None."""
    # TODO
    # 1) handle empty input
    if not user_town or not user_town.strip():
        return None
    # 2) exact lower-case match
    lower = user_town.strip().lower()
    if lower in town_to_row:
        return lower
    # 3) relaxed contains-match over valid_towns
    for name in valid_towns:
        if lower in name.lower() or name.lower() in lower:
            return name.lower()


In [12]:
# Test the town matching
print(match_town("Zürich"))  # Exact match
print(match_town("winterthur"))  # Relaxed match
print(match_town("Kloten"))  # Exact match
print(match_town("Fantasiestadt")) # No match

zürich
winterthur
kloten
None


## Step 3 - Extract Preferences From Natural Language

Use an LLM call similar to Week 1 `llm_calls`
Write the user prompts in German because the dataset contains Swiss place names such as `Zürich`.

Helpful structure for this step:
- give the model a short system/developer instruction,
- tell it to return strict JSON only,
- name the three required keys exactly: `rooms`, `area_m2`, `town`,
- tell it to use numbers for `rooms` and `area_m2`.

Example user input:
`Ich suche eine 3.5-Zimmer-Wohnung mit etwa 85 m2 in Winterthur.`

Ideal JSON shape:
```json
{"rooms": 3.5, "area_m2": 85, "town": "Winterthur"}
```

After the LLM call, still validate in Python that all three values exist and that `town` can be matched with `match_town(...)`.


In [15]:
def extract_preferences(user_text: str) -> dict:
    """Extract rooms, area_m2, and town from free text."""
    
    openai_client = OpenAI(api_key=LLM_API_KEY)
    
    system_prompt = """Du bist ein spezialisierter Assistent für Schweizer Wohnungssuche.

Deine Aufgabe: Extrahiere aus dem Benutzertext genau drei Informationen:
1. rooms – Anzahl Zimmer als Zahl (z.B. 3.5)
2. area_m2 – Wohnfläche in Quadratmetern als Zahl (z.B. 85)
3. town – Name der Schweizer Gemeinde als String (z.B. "Winterthur")

Regeln:
- Antworte ausschliesslich mit validem JSON, kein zusätzlicher Text.
- Verwende exakt diese drei Keys: rooms, area_m2, town
- rooms und area_m2 müssen numerische Werte sein, keine Strings.
- town muss ein Schweizer Ortsname sein, korrekt geschrieben.
- Falls eine Information im Text fehlt, setze den Wert auf null.

Beispiel-Input: "Ich suche eine 3.5-Zimmer-Wohnung mit etwa 85 m2 in Winterthur."
Beispiel-Output: {"rooms": 3.5, "area_m2": 85, "town": "Winterthur"}"""
    
    response = openai_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_text}
        ],
        temperature=0
    )
    
    raw = response.choices[0].message.content
    parsed = json.loads(raw)
    
    for key in ("rooms", "area_m2", "town"):
        if key not in parsed:
            raise ValueError(f"Missing key: {key}")
    
    matched = match_town(parsed["town"])
    if matched is None:
        raise ValueError(f"Town not found in dataset: {parsed['town']}")
    parsed["town"] = matched
    
    return parsed

In [16]:

result = extract_preferences("Ich suche eine 3.5-Zimmer-Wohnung mit etwa 85 m2 in Winterthur.")
print(result)

{'rooms': 3.5, 'area_m2': 85, 'town': 'winterthur'}


## Step 4 - Predict Monthly Rent

Use the loaded random forest model and the same seven input features shown in `apartment.ipynb` from the exercise where you created the model.


In [17]:
def predict_apartment_price(rooms: float, area_m2: float, town: str) -> float:
    """Predict monthly rent using the loaded random forest model."""
    
    matched = match_town(town)
    if matched is None:
        raise ValueError(f"Town not found in dataset: {town}")
    
    row = town_to_row[matched]
    
    features = np.array([[
        rooms,
        area_m2,
        row["pop"],
        row["pop_dens"],
        row["frg_pct"],
        row["emp"],
        row["tax_income"]
    ]])
    
    prediction = model.predict(features)[0]
    return round(prediction, 2)

In [21]:
#Test
price = predict_apartment_price(3.5, 85, "winterthur")
print(f"Geschätzte Miete: {price} CHF")

Geschätzte Miete: 2117.32 CHF


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


## Step 5 - Generate A User-Friendly Explanation

This second LLM step should not predict the price again. The model prediction already exists. The LLM should only explain the result in simple language.

A good explanation prompt should include:
- the structured preferences,
- the predicted rent in CHF,
- a request for a short German answer,
- one uncertainty or limitation note.

Ideal output shape:
```json
{"answer": "Für eine 3.5-Zimmer-Wohnung in Winterthur schÃ¤tzt das Modell rund 2800 CHF pro Monat. Die Schätzung orientiert sich an Wohnfläche und Ortsmerkmalen. Eine Unsicherheit ist, dass Zustand, Lage im Ort und Ausstattungsstandard im Modell nicht direkt enthalten sind."}
```


## Step 4 - Predict Monthly Rent

Use the loaded random forest model and the same seven input features shown in `apartment.ipynb` from the exercise where you created the model.


In [22]:
def generate_explanation(preferences: dict, prediction: float) -> str:
    """Generate a user-friendly German explanation of the prediction."""
    
    openai_client = OpenAI(api_key=LLM_API_KEY)
    
    system_prompt = """Du bist ein hilfreicher Wohnungsberater in der Schweiz.

Deine Aufgabe: Erkläre dem Benutzer die Mietpreis-Schätzung in einfachem Deutsch.

Regeln:
- Du bekommst die Wohnungswünsche und eine bereits berechnete Schätzung in CHF.
- Erkläre das Ergebnis in 2-3 Sätzen auf Deutsch.
- Erfinde keinen neuen Preis – verwende ausschliesslich den übergebenen Wert.
- Erwähne eine Unsicherheit oder Limitation des Modells (z.B. Zustand, Lage, Ausstattung fehlen).
- Antworte ausschliesslich mit validem JSON mit dem Key: answer

Beispiel-Output:
{"answer": "Für eine 3.5-Zimmer-Wohnung in Winterthur schätzt das Modell rund 2100 CHF pro Monat. Die Schätzung basiert auf Wohnfläche und Gemeindemerkmalen. Eine Unsicherheit ist, dass Zustand, Mikrolage und Ausstattung nicht im Modell enthalten sind."}"""

    user_prompt = (
        f"Wohnungswünsche: {json.dumps(preferences, ensure_ascii=False)}\n"
        f"Geschätzte Monatsmiete: {prediction} CHF"
    )
    
    response = openai_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0
    )
    
    raw = response.choices[0].message.content
    parsed = json.loads(raw)
    
    if "answer" not in parsed:
        raise ValueError("Missing key: answer")
    
    return parsed["answer"]

In [25]:
#Test
prefs = {"rooms": 3.5, "area_m2": 85, "town": "winterthur"}
explanation = generate_explanation(prefs, 2117.32)
print(explanation)

Für eine 3.5-Zimmer-Wohnung mit 85 m² in Winterthur schätzt das Modell die Monatsmiete auf 2117.32 CHF. Die Schätzung basiert auf den angegebenen Wohnungsdaten. Eine Unsicherheit ist, dass Zustand, genaue Lage und Ausstattung der Wohnung nicht berücksichtigt sind.


## Step 6 - End-To-End Pipeline

Suggested order inside `run_pipeline(...)`:
1. call `extract_preferences(...)`
2. call `predict_apartment_price(...)`
3. call `generate_explanation(...)`
4. return `(preferences, prediction, answer)`


In [26]:
def run_pipeline(user_text: str):
    """End-to-end pipeline: extract → predict → explain."""
    
    preferences = extract_preferences(user_text)
    prediction = predict_apartment_price(
        preferences["rooms"],
        preferences["area_m2"],
        preferences["town"]
    )
    explanation = generate_explanation(preferences, prediction)
    
    return preferences, prediction, explanation

In [27]:
#Test
prefs, price, answer = run_pipeline("Ich suche eine 3.5-Zimmer-Wohnung mit etwa 85 m2 in Winterthur.")
print(f"Extrahiert: {prefs}")
print(f"Preis: {price} CHF")
print(f"Antwort: {answer}")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


Extrahiert: {'rooms': 3.5, 'area_m2': 85, 'town': 'winterthur'}
Preis: 2117.32 CHF
Antwort: Für eine 3.5-Zimmer-Wohnung in Winterthur schätzt das Modell die Monatsmiete auf 2117.32 CHF. Die Schätzung basiert auf den angegebenen Wohnungsdaten. Eine Unsicherheit ist, dass Zustand, genaue Lage und Ausstattung der Wohnung nicht berücksichtigt sind.


## Step 7 - Test With Multiple Examples


In [30]:
test_inputs = [
    "Ich suche eine 3.5-Zimmer-Wohnung mit 85 m2 in Winterthur.",
    "Ich suche 2 Zimmer und etwa 55 m2 in Kloten.",
    "Ich brauche eine 4-Zimmer-Wohnung mit rund 110 m2 in Zürich.",
]

# TODO: loop over test_inputs and print results from run_pipeline(...)

for text in test_inputs:
    print(f"\nInput: {text}")
    prefs, price, answer = run_pipeline(text)
    print(f"Extrahiert: {prefs}")
    print(f"Preis: {price} CHF")
    print(f"Antwort: {answer}")
    print("-" * 60)


Input: Ich suche eine 3.5-Zimmer-Wohnung mit 85 m2 in Winterthur.


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


Extrahiert: {'rooms': 3.5, 'area_m2': 85, 'town': 'winterthur'}
Preis: 2117.32 CHF
Antwort: Für eine 3.5-Zimmer-Wohnung mit 85 m² in Winterthur schätzt das Modell die Monatsmiete auf 2117.32 CHF. Die Schätzung basiert auf den angegebenen Wohnungsdaten. Eine Unsicherheit ist, dass Zustand, genaue Lage und Ausstattung der Wohnung nicht berücksichtigt sind.
------------------------------------------------------------

Input: Ich suche 2 Zimmer und etwa 55 m2 in Kloten.


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


Extrahiert: {'rooms': 2, 'area_m2': 55, 'town': 'kloten'}
Preis: 1881.22 CHF
Antwort: Für eine 2-Zimmer-Wohnung mit 55 m² in Kloten schätzt das Modell rund 1881.22 CHF pro Monat. Die Schätzung basiert auf den angegebenen Wohnungsdaten. Eine Unsicherheit ist, dass Zustand, genaue Lage und Ausstattung der Wohnung nicht berücksichtigt sind.
------------------------------------------------------------

Input: Ich brauche eine 4-Zimmer-Wohnung mit rund 110 m2 in Zürich.


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


Extrahiert: {'rooms': 4, 'area_m2': 110, 'town': 'zürich'}
Preis: 4028.79 CHF
Antwort: Für eine 4-Zimmer-Wohnung mit 110 m² in Zürich schätzt das Modell rund 4028.79 CHF pro Monat. Die Schätzung basiert auf den angegebenen Eckdaten. Eine Unsicherheit ist, dass Zustand, genaue Lage und Ausstattung der Wohnung nicht berücksichtigt sind.
------------------------------------------------------------


## Step 8 - Notebook To app_student.py Then app.py

Copy your final function implementations into `app_student.py`:
- `extract_preferences`
- `predict_apartment_price`
- `generate_explanation`
- `run_pipeline`

Keep the saved model path as `random_forest_regression.pkl`.

Use `NOTEBOOK_TO_APP.md` as the deployment checklist.


## Submission Requirements

1. Working numeric prediction flow in notebook using the provided `.pkl` model
2. Working free-text input parsing to structured fields (LLM required)
3. Extraction prompt designed so the LLM returns strict JSON with `rooms`, `area_m2`, and `town`
4. Test inputs and prompts written in German so Swiss town names match the dataset reliably
5. Response text generated with an LLM and including one uncertainty/limitation
6. Transfer of notebook logic into `app_student.py`, then promote to `app.py`
7. No fallback path: missing key/API errors must remain visible
8. Short reflection (3-5 sentences): strengths, limits, responsible use
